In [1]:
import numpy as np
import os
import joblib
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report

In [2]:
DATA_DIR = 'processed_data'
MODEL_DIR = 'models'
RESULTS_DIR = 'results'

PARAM_GRID_ver1 = {
    'n_estimators': [100, 200, 300],       
    'max_depth': [3, 5, 7],                
    'learning_rate': [0.01, 0.1, 0.2],     
    'subsample': [0.8, 1.0]                
}

PARAM_GRID = {
    'n_estimators': [100, 200, 300, 400, 500],  
    'max_depth': [1, 2, 3, 5, 7, 9],            
    'learning_rate': [0.01, 0.1, 0.2, 0.3],     
    'subsample': [0.8, 1.0]                
}

def train_baseline(feature_file, label_file, group_file, model_name):
    print(f"\n" + "="*60)
    print(f"  TRAINING PIPELINE: {model_name}")
    print("="*60)

    print("1. Loading Data...")
    X_train = np.load(os.path.join(DATA_DIR, feature_file))
    y_train = np.load(os.path.join(DATA_DIR, label_file))
    groups_train = np.load(os.path.join(DATA_DIR, group_file))
    
    print(f"   Features: {X_train.shape}")
    print(f"   Labels:   {y_train.shape}")
    
    cv_splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    cv_folds = list(cv_splitter.split(X_train, groups_train))
    
    print(f"   Created {len(cv_folds)} validation folds based on Stratification Groups.")

    xgb = XGBClassifier(
        device='cuda',
        tree_method='hist', 
        use_label_encoder=False, 
        eval_metric='logloss', 
        random_state=42,
        n_jobs=1 
    )

    print(f"\n2. Starting Grid Search optimization...")
    
    grid_search = GridSearchCV(
        estimator=xgb,
        param_grid=PARAM_GRID,
        cv=cv_folds, 
        scoring='f1', 
        verbose=1,
        n_jobs=4     
    )
    
    grid_search.fit(X_train, y_train)
    
    print(f"\n3. Tuning Complete for {model_name}!")
    print(f"   ✅ Best Validation F1 Score: {grid_search.best_score_:.4f}")
    print(f"   ✅ Best Hyperparameters:     {grid_search.best_params_}")
    
    if not os.path.exists(MODEL_DIR): os.makedirs(MODEL_DIR)
    if not os.path.exists(RESULTS_DIR): os.makedirs(RESULTS_DIR)
    
    model_path = os.path.join(MODEL_DIR, f"{model_name}_best.pkl")
    joblib.dump(grid_search.best_estimator_, model_path)
    print(f"    Saved model to: {model_path}")
    
    results_df = pd.DataFrame([grid_search.best_params_])
    results_df['best_cv_f1_score'] = grid_search.best_score_
    csv_path = os.path.join(RESULTS_DIR, f"{model_name}_params.csv")
    results_df.to_csv(csv_path, index=False)
    print(f"    Saved parameters to: {csv_path}")

if __name__ == "__main__":
    train_baseline(
        feature_file='X_absa_train.npy', 
        label_file='y_train.npy', 
        group_file='groups_train.npy', 
        model_name='ABSA_Baseline'
    )

    train_baseline(
        feature_file='X_emo_train.npy', 
        label_file='y_train.npy', 
        group_file='groups_train.npy', 
        model_name='Emotion_Baseline'
    )


  TRAINING PIPELINE: ABSA_Baseline
1. Loading Data...
   Features: (1360, 768)
   Labels:   (1360,)
   Created 10 validation folds based on Stratification Groups.

2. Starting Grid Search optimization...
Fitting 10 folds for each of 240 candidates, totalling 2400 fits


C:\Users\User\Desktop\FYP_Coding\fyp_venv\lib\site-packages\xgboost\training.py:199: UserWarning: [19:58:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



3. Tuning Complete for ABSA_Baseline!
   ✅ Best Validation F1 Score: 0.8166
   ✅ Best Hyperparameters:     {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 400, 'subsample': 0.8}
    Saved model to: models\ABSA_Baseline_best.pkl
    Saved parameters to: results\ABSA_Baseline_params.csv

  TRAINING PIPELINE: Emotion_Baseline
1. Loading Data...
   Features: (1360, 768)
   Labels:   (1360,)
   Created 10 validation folds based on Stratification Groups.

2. Starting Grid Search optimization...
Fitting 10 folds for each of 240 candidates, totalling 2400 fits


C:\Users\User\Desktop\FYP_Coding\fyp_venv\lib\site-packages\xgboost\training.py:199: UserWarning: [23:13:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



3. Tuning Complete for Emotion_Baseline!
   ✅ Best Validation F1 Score: 0.7574
   ✅ Best Hyperparameters:     {'learning_rate': 0.3, 'max_depth': 5, 'n_estimators': 300, 'subsample': 0.8}
    Saved model to: models\Emotion_Baseline_best.pkl
    Saved parameters to: results\Emotion_Baseline_params.csv


# Testing

In [3]:
import numpy as np
import pandas as pd
import os
import pickle
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

DATA_DIR = 'processed_data'
MODEL_DIR = 'models'

def evaluate_saved_model(model_name, model_file, test_data_file):
    print(f"\n" + "="*60)
    print(f"🧪 EVALUATING: {model_name}")
    print("="*60)
    
    print(f"1. Loading Test Data ({test_data_file})...")
    try:
        X_test = np.load(os.path.join(DATA_DIR, test_data_file))
        y_test = np.load(os.path.join(DATA_DIR, 'y_test.npy'))
    except FileNotFoundError:
        print(f"❌ Error: Could not find data file '{test_data_file}' in '{DATA_DIR}'.")
        return

    model_path = os.path.join(MODEL_DIR, model_file)
    print(f"2. Loading Model from {model_path}...")
    
    try:
        with open(model_path, 'rb') as f:
            model = pickle.load(f)
        print("   ✅ Model Loaded Successfully")
    except FileNotFoundError:
        print(f"❌ Error: Model file '{model_file}' not found in '{MODEL_DIR}'.")
        return
    except Exception as e:
        print(f"❌ Error loading pickle: {e}")
        return

    print("3. Predicting on Test Set...")
    try:
        y_pred = model.predict(X_test)
    except Exception as e:
        print(f"❌ Prediction Error (Check if features match): {e}")
        return

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print("\n" + "-"*40)
    print(f"🏆 {model_name} TEST RESULTS")
    print("-" * 40)
    print(f"   Accuracy: {acc:.4f}")
    print(f"   F1 Score: {f1:.4f}")
    print("\n   --- Classification Report ---")
    print(classification_report(y_test, y_pred, digits=4))
    print("   --- Confusion Matrix ---")
    print(confusion_matrix(y_test, y_pred))

if __name__ == "__main__":
    evaluate_saved_model(
        model_name="ABSA Baseline",
        model_file="ABSA_Baseline_best.pkl",  
        test_data_file="X_absa_test.npy"      
    )

    evaluate_saved_model(
        model_name="Emotion Baseline",
        model_file="Emotion_Baseline_best.pkl",
        test_data_file="X_emo_test.npy"        
    )


🧪 EVALUATING: ABSA Baseline
1. Loading Test Data (X_absa_test.npy)...
2. Loading Model from models\ABSA_Baseline_best.pkl...
   ✅ Model Loaded Successfully
3. Predicting on Test Set...

----------------------------------------
🏆 ABSA Baseline TEST RESULTS
----------------------------------------
   Accuracy: 0.8375
   F1 Score: 0.8408

   --- Classification Report ---
              precision    recall  f1-score   support

           0     0.8522    0.8167    0.8340       120
           1     0.8240    0.8583    0.8408       120

    accuracy                         0.8375       240
   macro avg     0.8381    0.8375    0.8374       240
weighted avg     0.8381    0.8375    0.8374       240

   --- Confusion Matrix ---
[[ 98  22]
 [ 17 103]]

🧪 EVALUATING: Emotion Baseline
1. Loading Test Data (X_emo_test.npy)...
2. Loading Model from models\Emotion_Baseline_best.pkl...
   ✅ Model Loaded Successfully
3. Predicting on Test Set...

----------------------------------------
🏆 Emotion Baselin